In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from matplotlib import pyplot as plt 
from datetime import datetime, timedelta

from emu_renewal.inputs import get_var_target

In [ ]:
country = "DEU"
var_prop = get_var_target(country)
num_index = [t.timestamp() for t in var_prop.index]

In [ ]:
def cosine_function(t, start, end):
    period = end - start
    curve = lambda x: 0.5 * np.cos((x - start) * np.pi / period) + 0.5
    in_range = abs(t - start - period / 2.0) < period / 2.0
    conditions = [t <= start, in_range, t >= start + period]
    functions = [lambda x: 1.0, curve, lambda x: 0.0]
    return np.piecewise(t, conditions, functions)

In [ ]:
data_ends = [num_index[0], num_index[-1]]
params, params_covariance = curve_fit(cosine_function, num_index, var_prop, p0=data_ends)

In [ ]:
x_vals = np.linspace(*data_ends, 100)
fit_result = cosine_function(x_vals, *params)
fit_date_vals = [datetime(1970, 1, 1) + timedelta(seconds=t) for t in x_vals]
pd.Series(fit_result, index=fit_date_vals).plot()
var_prop.plot()